In [11]:
import h5py

# Path to your file (update filename as needed)
file_path = "/home/jovyan/shared/data/NSD/MTL_betas/subj01_all_mtl_sessions.h5"

# Open the file in read-only mode
with h5py.File(file_path, "r") as f:
    # List all top-level keys (these correspond to sessions)
    sessions = list(f.keys())
    print("Sessions:", sessions)
    
    # Pick a session, for example 'session01'
    session = 'session01'
    rois = list(f[session].keys())
    print(f"ROIs in {session}:", rois)
    
    # Load data for a specific ROI, e.g. 'roi_1'
    roi_data = f[session]['roi_1'][()]
    print(f"Shape of roi_1 data in {session}:", roi_data.shape)


Sessions: ['session01', 'session02', 'session03', 'session04', 'session05', 'session06', 'session07', 'session08', 'session09', 'session10', 'session11', 'session12', 'session13', 'session14', 'session15', 'session16', 'session17', 'session18', 'session19', 'session20', 'session21', 'session22', 'session23', 'session24', 'session25', 'session26', 'session27', 'session28', 'session29', 'session30', 'session31', 'session32', 'session33', 'session34', 'session35', 'session36', 'session37', 'session38', 'session39', 'session40']
ROIs in session01: ['roi_1', 'roi_10', 'roi_2', 'roi_3', 'roi_4', 'roi_5', 'roi_6', 'roi_7', 'roi_8', 'roi_9']
Shape of roi_1 data in session01: (750, 2163)


In [12]:
import h5py

file_path = "/home/jovyan/shared/data/NSD/MTL_betas/subj01_all_mtl_sessions.h5"

all_data = {}

with h5py.File(file_path, "r") as f:
    for session_key in f.keys():
        all_data[session_key] = {}
        for roi_key in f[session_key].keys():
            all_data[session_key][roi_key] = f[session_key][roi_key][()]


In [9]:
# This code:

# Loads the MTL ROI mask.
# Loads session-specific ROI beta values stored as 2D arrays (trials x voxels) per ROI.
# For a single trial (trial 0), reconstructs a full 3D volume by putting the beta values back into their original voxel locations according to the ROI mask.



import h5py
import numpy as np

# Load MTL mask (3D ROI labels)
mtl_mask_path = "/home/jovyan/cache/memoryNSD/subj01_MTL.nii.gz"
mtl_img = nib.load(mtl_mask_path)
MTL_data = mtl_img.get_fdata()

# Open .h5 file and load all ROIs for one session
file_path = "/home/jovyan/shared/data/NSD/MTL_betas/subj01_all_mtl_sessions.h5"
session_key = "session01"

with h5py.File(file_path, "r") as f:
    roi_data_dict = {}
    for roi_key in f[session_key].keys():
        # roi_key will be strings like 'roi_1', 'roi_2', ...
        roi_label = int(roi_key.split('_')[1])  # get label number as int
        roi_data_dict[roi_label] = f[session_key][roi_key][()]  # load full roi array: shape (n_trials, n_voxels)

# Now reinflate trial 0 for the entire MTL mask
trial_index = 0
full_volume = np.zeros(MTL_data.shape)

for roi_label, roi_betas in roi_data_dict.items():
    roi_indices = np.where(MTL_data == roi_label)
    trial_beta_vals = roi_betas[trial_index, :]  # 1D array for this trial and ROI
    
    full_volume[roi_indices] = trial_beta_vals

print("full volume shape:", full_volume.shape)


# import os
# import nibabel as nib
# import numpy as np
# import h5py

# # --- Step 1: Download the MTL mask ---
# def download_mtl_mask(subject, save_path="/home/jovyan/cache/memoryNSD/"):
#     subj_str = f"subj{subject:02d}"
#     url = f"https://natural-scenes-dataset.s3.amazonaws.com/nsddata/ppdata/{subj_str}/func1mm/roi/MTL.nii.gz"
#     file_path = os.path.join(save_path, f"{subj_str}_MTL.nii.gz")

#     os.makedirs(save_path, exist_ok=True)
#     os.system(f"wget -q -O {file_path} {url}")
#     print(f"Downloaded MTL mask: {file_path}")
#     return file_path

# # Download the mask for subject 01
# mtl_mask_path = download_mtl_mask(1)

# # --- Step 2: Load the corresponding 3D MTL mask ---
# mtl_img = nib.load(mtl_mask_path)
# MTL_data = mtl_img.get_fdata()
# roi_mask = MTL_data > 0
# mask_shape = MTL_data.shape

# # --- Step 3: Load 2D ROI data from the HDF5 file ---
# h5_path = "/home/jovyan/shared/data/NSD/MTL_betas/subj01_all_mtl_sessions.h5"

# with h5py.File(h5_path, "r") as f:
#     roi_data = f["session01"]["roi_1"][()]  # Shape: (750, N_voxels)

# print("Shape of trial_0_flat:", trial_0_flat.shape)
# print("Number of voxels in mask:", np.sum(roi_mask))

# # --- Step 4: Reinflate one trial back to 3D ---
# trial_0_flat = roi_data[0]  # 1D array of voxel values
# trial_0_volume = np.zeros(mask_shape)
# trial_0_volume[roi_mask] = trial_0_flat

# print("✅ Reinflated trial_0 volume shape:", trial_0_volume.shape)  # Should be (91, 109, 91)


full volume shape: (145, 186, 148)
